# 06 Trigram


In [ ]:
from urllib.request import urlopen
import csv
import io
import random
import unicodedata

import torch

url = "https://raw.githubusercontent.com/niyazikemer/turkce_isimler/main/turkce_isim.csv"
text = urlopen(url).read().decode("utf-8-sig")
rows = csv.DictReader(io.StringIO(text))
words = sorted(set(unicodedata.normalize("NFC", row["name"].strip().lower()) for row in rows))
words = [w for w in words if w and w.isalpha()]

chars = sorted(list(set("".join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i: s for s, i in stoi.items()}
V = len(stoi)

random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))
train = words[:n1]
dev = words[n1:n2]
test = words[n2:]


def make_counts(data, order):
    rows = V if order == 2 else V * V
    N = torch.zeros((rows, V), dtype=torch.int32)
    for w in data:
        chs = ["."] * (order - 1) + list(w) + ["."]
        for i in range(len(chs) - order + 1):
            if order == 2:
                row = stoi[chs[i]]
            else:
                row = stoi[chs[i]] * V + stoi[chs[i + 1]]
            N[row, stoi[chs[i + order - 1]]] += 1
    return N


def get_loss(data, P, order):
    total = 0.0
    n = 0
    for w in data:
        chs = ["."] * (order - 1) + list(w) + ["."]
        for i in range(len(chs) - order + 1):
            if order == 2:
                row = stoi[chs[i]]
            else:
                row = stoi[chs[i]] * V + stoi[chs[i + 1]]
            target = stoi[chs[i + order - 1]]
            total += -torch.log(P[row, target]).item()
            n += 1
    return total / n


def tune(order):
    N = make_counts(train, order)
    best = None
    for alpha in [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]:
        P = N.float() + alpha
        P /= P.sum(1, keepdim=True)
        loss = get_loss(dev, P, order)
        print(order, alpha, loss)
        if best is None or loss < best[0]:
            best = (loss, alpha, P)
    return best


bigram = tune(2)
trigram = tune(3)

print("bigram test:", get_loss(test, bigram[2], 2), "alpha:", bigram[1])
print("trigram test:", get_loss(test, trigram[2], 3), "alpha:", trigram[1])


def sample(P, order, seed):
    g = torch.Generator().manual_seed(seed)
    result = []
    for i in range(10):
        context = [0] * (order - 1)
        out = []
        while True:
            row = context[0] if order == 2 else context[0] * V + context[1]
            ix = torch.multinomial(P[row], 1, replacement=True, generator=g).item()
            if ix == 0:
                break
            out.append(itos[ix])
            context = (context + [ix])[-(order - 1):]
        result.append("".join(out))
    return result


print("bigram:", sample(bigram[2], 2, 42))
print("trigram:", sample(trigram[2], 3, 42))
